# Notebook 05 - Planificateur de Repas : Data Fusion LINQ + Théorème Hiérarchique

**Navigation** : [Index Z3](./README.md) | [Index SymbolicAI](../../README.md) | [Index général](../../../README.md)

## Objectifs d'apprentissage

- Comprendre le **théorème hiérarchique** : modèles Z3 avec objets imbriqués
- Pratiquer la **fusion de données** entre sources externes (nutrition) et variables Z3
- Modéliser un **planificateur de repas** équilibré avec contraintes nutritionnelles
- Explorer l'**optimisation** (minimiser un cout sous contraintes)

## Prerequis

- Notebooks 01 (Introduction Z3.Linq) et 04 (Nested Arrays)
- Concepts : types generiques C#, LINQ, contraintes lineaires

**Duree estimee** : 30-40 minutes

In [1]:
#r "../Z3.Linq/.deploy/Microsoft.Z3.dll"
#r "../Z3.Linq/.deploy/ExpressionUtils.dll"
#r "../Z3.Linq/.deploy/Z3.Linq.dll"
using Z3.Linq;
using Microsoft.Z3;
using System;
using System.Collections.Generic;
using System.Linq;
Console.WriteLine("Imports OK : Z3.Linq (.deploy, CollectionHandling câblé), Microsoft.Z3, System.Linq");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Imports OK : Z3.Linq (.deploy, CollectionHandling câblé), Microsoft.Z3, System.Linq


---

## 1. Contexte : Le théorème hiérarchique

Dans les notebooks précédents, nous avons utilisé `Symbols<T1, T2, ...>` pour créer des variables Z3 plates. Le **théorème hiérarchique** generalise cette approche :

- Un objet parent (ex: `Meal`) contient des references vers des objets enfants (ex: `Course`)
- Chaque niveau de la hiérarchie possède ses propres contraintes
- Le solveur Z3 résout le système complet de manière cohérente

### Analogie

```
Meal (contraintes globales : calories totales, budget)
  |-- Entree (contraintes : type, taille)
  |-- Plat principal (contraintes : proteines minimales)
  |-- Dessert (contraintes : sucre maximal)
```

La **fusion de données** consiste a injecter des données externes (table nutritionnelle) dans les contraintes du solveur.

## 2. Base de données nutritionnelle (data fusion)

Nous definissons une base de données d'aliments avec leurs propriétés nutritionnelles. Ces données seront utilisees par le solveur pour sélectionner des combinaisons valides.

In [2]:
// Base de données nutritionnelle : fusion de données statiques avec le solveur
public class FoodItem
{
    public string Name { get; set; }
    public int Calories { get; set; }      // kcal par portion
    public int Protein { get; set; }       // grammes
    public int Carbs { get; set; }         // grammes
    public int Fat { get; set; }           // grammes
    public int Cost { get; set; }          // prix en centimes
    public string Category { get; set; }   // entree, plat, dessert
}

var foodDatabase = new List<FoodItem>
{
    // Entrees
    new FoodItem { Name = "Salade verte",    Calories = 80,  Protein = 2,  Carbs = 8,   Fat = 4,  Cost = 300, Category = "entree" },
    new FoodItem { Name = "Soupe de legumes", Calories = 120, Protein = 4,  Carbs = 18,  Fat = 3,  Cost = 250, Category = "entree" },
    new FoodItem { Name = "Carpaccio",        Calories = 150, Protein = 15, Carbs = 2,   Fat = 8,  Cost = 600, Category = "entree" },
    // Plats principaux
    new FoodItem { Name = "Poulet roti",      Calories = 350, Protein = 30, Carbs = 5,   Fat = 20, Cost = 500, Category = "plat" },
    new FoodItem { Name = "Saumon grille",    Calories = 300, Protein = 28, Carbs = 0,   Fat = 18, Cost = 800, Category = "plat" },
    new FoodItem { Name = "Pates bolognaise", Calories = 450, Protein = 18, Carbs = 55,  Fat = 14, Cost = 350, Category = "plat" },
    new FoodItem { Name = "Risotto",          Calories = 400, Protein = 10, Carbs = 60,  Fat = 12, Cost = 400, Category = "plat" },
    // Desserts
    new FoodItem { Name = "Fruit frais",      Calories = 70,  Protein = 1,  Carbs = 15,  Fat = 0,  Cost = 200, Category = "dessert" },
    new FoodItem { Name = "Mousse au chocolat",Calories = 250, Protein = 4,  Carbs = 28,  Fat = 14, Cost = 350, Category = "dessert" },
    new FoodItem { Name = "Yaourt nature",    Calories = 60,  Protein = 5,  Carbs = 6,   Fat = 2,  Cost = 150, Category = "dessert" },
};

Console.WriteLine($"Base de données chargée : {foodDatabase.Count} aliments");
Console.WriteLine(string.Join("\n", foodDatabase.GroupBy(f => f.Category)
    .Select(g => $"  {g.Key} : {g.Count()} items")));

Base de données chargée : 10 aliments


  entree : 3 items
  plat : 4 items
  dessert : 3 items


### Interpretation

La base contient 10 aliments répartis en 3 catégories (entree, plat, dessert). Chaque aliment a des propriétés nutritionnelles et un cout. Le solveur devra sélectionner exactement **1 entree + 1 plat + 1 dessert** qui satisfont les contraintes globales.

## 3. Approche 1 : Sélection par index avec contraintes lineaires

La première approche utilisé des variables entieres pour sélectionner un aliment dans chaque catégorie. Les contraintes sont exprimees lineairement en fonction de l'index.

In [3]:
// Approche index-based : chaque variable represente l'index d'un aliment
// Les contraintes Z3.Linq doivent utiliser des expressions lineaires sur les variables

var entrees = foodDatabase.Where(f => f.Category == "entree").ToList();
var plats = foodDatabase.Where(f => f.Category == "plat").ToList();
var desserts = foodDatabase.Where(f => f.Category == "dessert").ToList();

using (var ctx = new Z3Context())
{
    // Variables : index de l'aliment choisi dans chaque catégorie
    // Contraintes de bornes avec expressions lineaires (pattern Z3.Linq)
    var theorem = from meal in ctx.NewTheorem<Symbols<int, int, int>>()
        where meal.X1 >= 0 && meal.X1 - 3 <= 0   // 0 <= X1 <= 2 (3 entrees)
        where meal.X2 >= 0 && meal.X2 - 4 <= 0   // 0 <= X2 <= 3 (4 plats)
        where meal.X3 >= 0 && meal.X3 - 3 <= 0   // 0 <= X3 <= 2 (3 desserts)
        select meal;

    var solution = theorem.Solve();
    
    if (solution != null)
    {
        var e = entrees[solution.X1];
        var p = plats[solution.X2];
        var d = desserts[solution.X3];
        int totalCal = e.Calories + p.Calories + d.Calories;
        int totalCost = e.Cost + p.Cost + d.Cost;
        
        Console.WriteLine("Menu trouve (sans contraintes nutritionnelles) :");
        Console.WriteLine($"  Entree  : {e.Name} ({e.Calories} kcal)");
        Console.WriteLine($"  Plat    : {p.Name} ({p.Calories} kcal)");
        Console.WriteLine($"  Dessert : {d.Name} ({d.Calories} kcal)");
        Console.WriteLine($"  Total   : {totalCal} kcal, {totalCost/100.0:F2} euros");
    }
    else
    {
        Console.WriteLine("Aucune solution trouvee");
    }
}

Menu trouve (sans contraintes nutritionnelles) :


  Entree  : Salade verte (80 kcal)


  Plat    : Poulet roti (350 kcal)


  Dessert : Fruit frais (70 kcal)


  Total   : 500 kcal, 10,00 euros


### Interpretation

Sans contraintes nutritionnelles, le solveur trouve la première combinaison valide (index 0, 0, 0). Le résultat est trivialement la première entree, le premier plat et le premier dessert de la base. Nous devons ajouter des contraintes pour obtenir un menu équilibré.

## 4. Approche 2 : Contraintes nutritionnelles avec énumération

La contrainte Z3.Linq fonctionne avec des expressions lineaires. Pour injecter les données nutritionnelles, nous énumérons les combinaisons possibles et posons des contraintes sur les totaux.

**Strategie** : generer toutes les combinaisons possibles, puis filtrer avec Z3 sur les contraintes globales (calories, proteines, budget).

In [4]:
// Approche par énumération + filtrage Z3
// Objectif : trouver un menu avec 600-900 kcal, >= 25g proteines, cout <= 12 euros

int minCalories = 600, maxCalories = 900;
int minProtein = 25;
int maxCostCents = 1200; // 12 euros

var validMenus = new List<string>();

// Énumération des combinaisons + vérification des contraintes
foreach (var e in entrees)
    foreach (var p in plats)
        foreach (var d in desserts)
        {
            int cal = e.Calories + p.Calories + d.Calories;
            int prot = e.Protein + p.Protein + d.Protein;
            int cost = e.Cost + p.Cost + d.Cost;
            
            if (cal >= minCalories && cal <= maxCalories 
                && prot >= minProtein && cost <= maxCostCents)
            {
                validMenus.Add($"{e.Name} + {p.Name} + {d.Name} = {cal} kcal, {prot}g prot, {cost/100.0:F2} EUR");
            }
        }

Console.WriteLine($"Menus valides (600-900 kcal, >= 25g prot, <= 12 EUR) : {validMenus.Count}/{entrees.Count * plats.Count * desserts.Count} combinaisons");
Console.WriteLine();
foreach (var menu in validMenus.Take(8))
    Console.WriteLine($"  {menu}");
if (validMenus.Count > 8)
    Console.WriteLine($"  ... et {validMenus.Count - 8} autres");

Menus valides (600-900 kcal, >= 25g prot, <= 12 EUR) : 8/36 combinaisons


  Salade verte + Poulet roti + Mousse au chocolat = 680 kcal, 36g prot, 11,50 EUR


  Soupe de legumes + Poulet roti + Mousse au chocolat = 720 kcal, 38g prot, 11,00 EUR


  Soupe de legumes + Pates bolognaise + Mousse au chocolat = 820 kcal, 26g prot, 9,50 EUR


  Soupe de legumes + Pates bolognaise + Yaourt nature = 630 kcal, 27g prot, 7,50 EUR


  Carpaccio + Pates bolognaise + Fruit frais = 670 kcal, 34g prot, 11,50 EUR


  Carpaccio + Pates bolognaise + Yaourt nature = 660 kcal, 38g prot, 11,00 EUR


  Carpaccio + Risotto + Fruit frais = 620 kcal, 26g prot, 12,00 EUR


  Carpaccio + Risotto + Yaourt nature = 610 kcal, 30g prot, 11,50 EUR


### Interpretation

L'énumération exhaustive fonctionne car l'espace est petit (3 x 4 x 3 = 36 combinaisons). Pour de grands catalogues, le solveur Z3 est préférable car il explore intelligemment l'espace de recherche plutôt que de tout énumérer.

## 5. Approche 3 : Modèle Z3 avec variables booléennes (sélection)

Pour modéliser la sélection d'aliments directement dans Z3, nous utilisons des **variables booléennes** : chaque aliment est soit sélectionné (`true`) soit non sélectionné (`false`).

### Principe

- Une variable booléenne par aliment
- Exactement 1 entree + 1 plat + 1 dessert sélectionnés
- Contraintes nutritionnelles sur la somme ponderee

In [5]:
// Approche booléenne Z3 : chaque aliment est une variable bool
// Avantage : le solveur explore l'espace, pas d'énumération

using (var z3ctx = new Context())
{
    var solver = z3ctx.MkSolver();
    
    // Variables booléennes : une par aliment
    var selections = foodDatabase
        .Select((food, i) => new { Food = food, Var = (BoolExpr)z3ctx.MkConst($"sel_{i}_{food.Name}", z3ctx.BoolSort) })
        .ToList();
    
    // Contrainte 1 : exactement 1 entree sélectionnée
    var entreeVars = selections.Where(s => s.Food.Category == "entree").Select(s => (BoolExpr)s.Var).ToArray();
    solver.Add(z3ctx.MkOr(entreeVars));           // au moins 1
    foreach (var v1 in entreeVars)                 // au plus 1 (pairwise exclusion)
        foreach (var v2 in entreeVars)
            if (v1 != v2)
                solver.Add(z3ctx.MkImplies(v1, z3ctx.MkNot(v2)));
    
    // Contrainte 2 : exactement 1 plat sélectionné
    var platVars = selections.Where(s => s.Food.Category == "plat").Select(s => (BoolExpr)s.Var).ToArray();
    solver.Add(z3ctx.MkOr(platVars));
    foreach (var v1 in platVars)
        foreach (var v2 in platVars)
            if (v1 != v2)
                solver.Add(z3ctx.MkImplies(v1, z3ctx.MkNot(v2)));
    
    // Contrainte 3 : exactement 1 dessert sélectionné
    var dessertVars = selections.Where(s => s.Food.Category == "dessert").Select(s => (BoolExpr)s.Var).ToArray();
    solver.Add(z3ctx.MkOr(dessertVars));
    foreach (var v1 in dessertVars)
        foreach (var v2 in dessertVars)
            if (v1 != v2)
                solver.Add(z3ctx.MkImplies(v1, z3ctx.MkNot(v2)));
    
    // Contraintes nutritionnelles lineaires (implication conditionnelle)
    IntExpr totalCal = (IntExpr)selections.Aggregate((IntExpr)z3ctx.MkInt(0), 
        (acc, s) => (IntExpr)z3ctx.MkAdd(acc, (ArithExpr)z3ctx.MkITE(s.Var, z3ctx.MkInt(s.Food.Calories), z3ctx.MkInt(0))));
    
    IntExpr totalProt = (IntExpr)selections.Aggregate((IntExpr)z3ctx.MkInt(0), 
        (acc, s) => (IntExpr)z3ctx.MkAdd(acc, (ArithExpr)z3ctx.MkITE(s.Var, z3ctx.MkInt(s.Food.Protein), z3ctx.MkInt(0))));
    
    IntExpr totalCost = (IntExpr)selections.Aggregate((IntExpr)z3ctx.MkInt(0), 
        (acc, s) => (IntExpr)z3ctx.MkAdd(acc, (ArithExpr)z3ctx.MkITE(s.Var, z3ctx.MkInt(s.Food.Cost), z3ctx.MkInt(0))));
    
    solver.Add(z3ctx.MkGe(totalCal, z3ctx.MkInt(600)));
    solver.Add(z3ctx.MkLe(totalCal, z3ctx.MkInt(900)));
    solver.Add(z3ctx.MkGe(totalProt, z3ctx.MkInt(25)));
    solver.Add(z3ctx.MkLe(totalCost, z3ctx.MkInt(1200)));
    
    Console.WriteLine($"Contraintes : {solver.NumAssertions} assertions");
    
    var result = solver.Check();
    Console.WriteLine($"Résultat solve : {result}");
    
    if (result == Status.SATISFIABLE)
    {
        var model = solver.Model;
        Console.WriteLine("\nMenu équilibré trouve :");
        int cal = 0, prot = 0, cost = 0;
        foreach (var s in selections)
        {
            var val = model.Eval(s.Var, true);
            if (val.BoolValue == Z3_lbool.Z3_L_TRUE)
            {
                Console.WriteLine($"  [{s.Food.Category}] {s.Food.Name} : {s.Food.Calories} kcal, {s.Food.Protein}g prot, {s.Food.Cost/100.0:F2} EUR");
                cal += s.Food.Calories;
                prot += s.Food.Protein;
                cost += s.Food.Cost;
            }
        }
        Console.WriteLine($"  --- Total : {cal} kcal, {prot}g proteines, {cost/100.0:F2} EUR ---");
    }
    else
    {
        Console.WriteLine("Aucun menu ne satisfait les contraintes.");
    }
}

Contraintes : 31 assertions


Résultat solve : SATISFIABLE



Menu équilibré trouve :


  [entree] Soupe de legumes : 120 kcal, 4g prot, 2,50 EUR


  [plat] Pates bolognaise : 450 kcal, 18g prot, 3,50 EUR


  [dessert] Mousse au chocolat : 250 kcal, 4g prot, 3,50 EUR


  --- Total : 820 kcal, 26g proteines, 9,50 EUR ---


### Interpretation

L'approche booléenne utilisé **`MkITE` (If-Then-Else)** pour conditionnellement ajouter la contribution nutritionnelle de chaque aliment. C'est la clé technique de la **data fusion** : les données statiques (calories, proteines, cout) sont injectees dans les expressions Z3 via `MkITE`.

| Technique | Role |
|-----------|------|
| `BoolExpr` par aliment | Sélection binaire |
| `MkOr` + exclusion mutuelle | Exactement 1 par catégorie |
| `MkITE(sel, valeur, 0)` | Contribution conditionnelle |
| Somme des `MkITE` | Total nutritionnel global |

## 6. Optimisation : menu le moins cher (dichotomie)

Le solveur Z3 (`MkSolver`) trouve une solution satisfaisante, mais pas forcement optimale. Pour trouver le menu **le moins cher**, nous utilisons une **recherche par dichotomie** :

1. On fixe une borne superieure sur le cout
2. On demande au solveur s'il existe un menu dans ce budget
3. Si oui, on essaie un budget plus faible ; si non, on augmente
4. On répète jusqu'a trouver le minimum exact

Cette technique (binary search on objective) est un pattern classique en optimisation par solveur SAT/SMT.

In [6]:
// Optimisation : trouver le menu équilibré le MOINS CHER
// On utilisé le solver avec exploration itérative : on ajoute une contrainte
// de cout de plus en plus strict jusqu'a ce que le probleme devienne UNSAT

using (var z3ctx = new Context())
{
    var solver = z3ctx.MkSolver();
    
    // Variables booléennes
    var selections = foodDatabase
        .Select((food, i) => new { Food = food, Var = (BoolExpr)z3ctx.MkConst($"sel_{i}_{food.Name}", z3ctx.BoolSort) })
        .ToList();
    
    // Exactement 1 par catégorie
    var entreeVars = selections.Where(s => s.Food.Category == "entree").Select(s => (BoolExpr)s.Var).ToArray();
    solver.Add(z3ctx.MkOr(entreeVars));
    foreach (var v1 in entreeVars)
        foreach (var v2 in entreeVars)
            if (!v1.Equals(v2))
                solver.Add(z3ctx.MkImplies(v1, z3ctx.MkNot(v2)));
    
    var platVars = selections.Where(s => s.Food.Category == "plat").Select(s => (BoolExpr)s.Var).ToArray();
    solver.Add(z3ctx.MkOr(platVars));
    foreach (var v1 in platVars)
        foreach (var v2 in platVars)
            if (!v1.Equals(v2))
                solver.Add(z3ctx.MkImplies(v1, z3ctx.MkNot(v2)));
    
    var dessertVars = selections.Where(s => s.Food.Category == "dessert").Select(s => (BoolExpr)s.Var).ToArray();
    solver.Add(z3ctx.MkOr(dessertVars));
    foreach (var v1 in dessertVars)
        foreach (var v2 in dessertVars)
            if (!v1.Equals(v2))
                solver.Add(z3ctx.MkImplies(v1, z3ctx.MkNot(v2)));
    
    // Contraintes nutritionnelles
    IntExpr totalCal = (IntExpr)selections.Aggregate((IntExpr)z3ctx.MkInt(0), 
        (acc, s) => (IntExpr)z3ctx.MkAdd(acc, (ArithExpr)z3ctx.MkITE(s.Var, z3ctx.MkInt(s.Food.Calories), z3ctx.MkInt(0))));
    IntExpr totalProt = (IntExpr)selections.Aggregate((IntExpr)z3ctx.MkInt(0), 
        (acc, s) => (IntExpr)z3ctx.MkAdd(acc, (ArithExpr)z3ctx.MkITE(s.Var, z3ctx.MkInt(s.Food.Protein), z3ctx.MkInt(0))));
    IntExpr totalCost = (IntExpr)selections.Aggregate((IntExpr)z3ctx.MkInt(0), 
        (acc, s) => (IntExpr)z3ctx.MkAdd(acc, (ArithExpr)z3ctx.MkITE(s.Var, z3ctx.MkInt(s.Food.Cost), z3ctx.MkInt(0))));
    
    solver.Add(z3ctx.MkGe(totalCal, z3ctx.MkInt(600)));
    solver.Add(z3ctx.MkLe(totalCal, z3ctx.MkInt(900)));
    solver.Add(z3ctx.MkGe(totalProt, z3ctx.MkInt(25)));
    solver.Add(z3ctx.MkLe(totalCost, z3ctx.MkInt(1200)));
    
    // Recherche du cout minimum par dichotomie
    int lo = 0, hi = 1200, bestCost = -1;
    Status bestResult = Status.UNSATISFIABLE;
    
    while (lo <= hi)
    {
        solver.Push();
        int mid = (lo + hi) / 2;
        solver.Add(z3ctx.MkLe(totalCost, z3ctx.MkInt(mid)));
        var r = solver.Check();
        solver.Pop();
        
        if (r == Status.SATISFIABLE)
        {
            bestCost = mid;
            bestResult = r;
            hi = mid - 1;  // essayer moins cher
        }
        else
        {
            lo = mid + 1;  // besoin de plus de budget
        }
    }
    
    // Résoudre avec le meilleur cout trouve
    solver.Add(z3ctx.MkLe(totalCost, z3ctx.MkInt(bestCost)));
    var result = solver.Check();
    Console.WriteLine($"Optimisation (dichotomie) : cout minimum = {bestCost/100.0:F2} EUR, résultat = {result}");
    
    if (result == Status.SATISFIABLE)
    {
        var model = solver.Model;
        Console.WriteLine("\nMenu optimal (cout minimum) :");
        int cal = 0, prot = 0, cost = 0;
        foreach (var s in selections)
        {
            var val = model.Eval(s.Var, true);
            if (val.BoolValue == Z3_lbool.Z3_L_TRUE)
            {
                Console.WriteLine($"  [{s.Food.Category}] {s.Food.Name} : {s.Food.Calories} kcal, {s.Food.Protein}g prot, {s.Food.Cost/100.0:F2} EUR");
                cal += s.Food.Calories;
                prot += s.Food.Protein;
                cost += s.Food.Cost;
            }
        }
        Console.WriteLine($"  === Total : {cal} kcal, {prot}g proteines, {cost/100.0:F2} EUR ===");
    }
    else
    {
        Console.WriteLine("Aucun menu optimal trouve.");
    }
}

Optimisation (dichotomie) : cout minimum = 7,50 EUR, résultat = SATISFIABLE



Menu optimal (cout minimum) :


  [entree] Soupe de legumes : 120 kcal, 4g prot, 2,50 EUR


  [plat] Pates bolognaise : 450 kcal, 18g prot, 3,50 EUR


  [dessert] Yaourt nature : 60 kcal, 5g prot, 1,50 EUR


  === Total : 630 kcal, 27g proteines, 7,50 EUR ===


### Interpretation

La **dichotomie sur le cout** est un pattern classique en optimisation SAT/SMT. Plutôt que d'utiliser `MkOptimizer` (qui peut etre instable dans certains bindings), on utilisé `solver.Push()`/`solver.Pop()` pour tester différentes bornes de cout sans reconstruire le solver a chaque fois.

| Aspect | Solve simple | Dichotomie |
|--------|-------------|------------|
| Objectif | Première solution | Solution optimale |
| Cout | Arbitraire | Minimum garanti |
| Appels solver | 1 | O(log(budget_max)) |
| Stabilite | Toujours | Toujours |

---

## 7. Théorème hiérarchique : plan hebdomadaire en `int[][]`

Jusqu'ici, nous avons manipulé des variables **plates** : des index isolés (section 3) ou un booléen par aliment (section 5). Pour planifier **plusieurs repas simultanément** — un plan sur N jours —, l'abstraction naturelle est un **tableau imbriqué** : une ligne par jour, une colonne par créneau (entrée, plat, dessert). C'est précisément ce que le fork `Z3.Linq` résout via le **théorème hiérarchique** sur `int[][]`.

Le même mécanisme qui résout une grille de Sudoku (`Cells[i][j]`, notebook 04) structure ici un plan de repas `Plan[jour][créneau]`. Les contraintes **structurelles** — catégorie imposée par créneau, non-répétition entre jours, montée en gamme — s'expriment en **pur LINQ**, sans descendre dans le Microsoft.Z3 brut.

### Modèle

```
Plan[0] = [entrée_j1, plat_j1, dessert_j1]   (indices dans foodDatabase)
Plan[1] = [entrée_j2, plat_j2, dessert_j2]
Plan[2] = [entrée_j3, plat_j3, dessert_j3]
```

Chaque cellule est un **entier = index d'un aliment**. Les aliments étant groupés par catégorie dans `foodDatabase` (entrées, puis plats, puis desserts), on impose des **bornes d'index** par créneau pour forcer la bonne catégorie.

In [7]:
// Théorème hiérarchique : plan de repas sur 3 jours, modélisé comme int[][] (jours x créneaux).
// Plan[j][0] = index de l'entrée, Plan[j][1] = index du plat, Plan[j][2] = index du dessert.
// foodDatabase est groupé par catégorie -> on calcule les bornes d'index de chaque catégorie.

public class WeeklyPlan
{
    public int[][] Plan { get; set; } = new int[3][];
    public WeeklyPlan() { for (int i = 0; i < 3; i++) Plan[i] = new int[3]; }
}

// Bornes d'index par catégorie (calculées depuis la base, pas codées en dur)
int entLo = foodDatabase.FindIndex(f => f.Category == "entree");
int entHi = foodDatabase.FindLastIndex(f => f.Category == "entree");
int pltLo = foodDatabase.FindIndex(f => f.Category == "plat");
int pltHi = foodDatabase.FindLastIndex(f => f.Category == "plat");
int desLo = foodDatabase.FindIndex(f => f.Category == "dessert");
int desHi = foodDatabase.FindLastIndex(f => f.Category == "dessert");
Console.WriteLine($"Catégories -> entrées [{entLo},{entHi}], plats [{pltLo},{pltHi}], desserts [{desLo},{desHi}]");

using (var ctx = new Z3Context())
{
    var theorem = ctx.NewTheorem<WeeklyPlan>();

    // Hiérarchie créneau -> catégorie : chaque créneau pointe vers sa catégorie d'aliments.
    for (int j = 0; j < 3; j++)
    {
        int jj = j;
        theorem = theorem.Where(p => p.Plan[jj][0] >= entLo && p.Plan[jj][0] <= entHi);
        theorem = theorem.Where(p => p.Plan[jj][1] >= pltLo && p.Plan[jj][1] <= pltHi);
        theorem = theorem.Where(p => p.Plan[jj][2] >= desLo && p.Plan[jj][2] <= desHi);
    }

    // Montée en gamme : plats strictement croissants sur la semaine (permutation ordonnée).
    theorem = theorem.Where(p => p.Plan[0][1] < p.Plan[1][1]);
    theorem = theorem.Where(p => p.Plan[1][1] < p.Plan[2][1]);

    // Diversité : pas le même dessert deux jours d'affilée.
    theorem = theorem.Where(p => p.Plan[0][2] != p.Plan[1][2]);
    theorem = theorem.Where(p => p.Plan[1][2] != p.Plan[2][2]);

    var sw = System.Diagnostics.Stopwatch.StartNew();
    var plan = theorem.Solve();
    sw.Stop();

    Console.WriteLine($"\nThéorème hiérarchique résolu en pur LINQ int[][] : {sw.Elapsed.TotalMilliseconds:F1} ms\n");
    for (int j = 0; j < 3; j++)
    {
        var e = foodDatabase[plan.Plan[j][0]];
        var pl = foodDatabase[plan.Plan[j][1]];
        var d = foodDatabase[plan.Plan[j][2]];
        Console.WriteLine($"Jour {j+1} : {e.Name,-18} | {pl.Name,-18} | {d.Name,-20}");
        Console.WriteLine($"        {e.Calories,3} + {pl.Calories,3} + {d.Calories,3} = {e.Calories+pl.Calories+d.Calories,3} kcal   ({e.Cost+pl.Cost+d.Cost,4} centimes)");
    }
}

Catégories -> entrées [0,2], plats [3,6], desserts [7,9]



Théorème hiérarchique résolu en pur LINQ int[][] : 26,8 ms



Jour 1 : Carpaccio          | Poulet roti        | Fruit frais         


        150 + 350 +  70 = 570 kcal   (1300 centimes)


Jour 2 : Salade verte       | Saumon grille      | Mousse au chocolat  


         80 + 300 + 250 = 630 kcal   (1450 centimes)


Jour 3 : Soupe de legumes   | Pates bolognaise   | Yaourt nature       


        120 + 450 +  60 = 630 kcal   ( 750 centimes)


### Interpretation — le domaine d'élégance du théorème hiérarchique

Le plan hebdomadaire est résolu en **pur LINQ** sur `int[][]` : **zéro** appel à `MkSolver`/`MkITE`/`MkOr`. C'est l'aboutissement du notebook — la même API qui résout une grille de Sudoku (notebook 04) structure un plan de repas multi-jours. Les trois familles de contraintes ci-dessous sont toutes **linéaires sur les valeurs d'index**, donc natives en LINQ :

| Contrainte | Expression LINQ | Nature |
|------------|-----------------|--------|
| Catégorie par créneau | `p.Plan[j][k] >= lo && p.Plan[j][k] <= hi` | Bornes linéaires |
| Montée en gamme (plats) | `p.Plan[0][1] < p.Plan[1][1]` | Ordre strict |
| Diversité (desserts) | `p.Plan[j][2] != p.Plan[j+1][2]` | Inégalité |

**Frontière honnête.** Ces contraintes portent sur la *position* des choix (les index eux-mêmes). Dès qu'on veut pondérer par un **attribut** de l'aliment — `foodDatabase[index].Calories`, un *lookup* — l'expression devient **non-linéaire** et le pur LINQ ne suffit plus. C'est exactement ce qui justifie les sections 4 à 6 (`MkITE`, énumération, dichotomie) : aucune approche n'est « meilleure » dans l'absolu, chacune a son domaine.

**Leçon SMT.** Le choix de modélisation se décide sur la *nature des contraintes* :

- **Structurelles** (portent sur la position/valeur des choix) → `int[][]` LINQ, concis et lisible.
- **Pondérées** (portent sur des attributs externes des choix) → `BoolExpr` + `MkITE`.

> **Corpus RecipeML + table Ciqual.** Notre `foodDatabase` compte 10 aliments — un sous-ensemble pédagogique. À l'échelle, le catalogue se construirait par **fusion de deux sources** — exactement le geste de la section 5 : **RecipeML** (standard XML de milliers de recettes) fournit la composition de chaque plat *avec les quantités* d'ingrédients — ce qui manquait, justement, aux menus de cantine bruts ; la **table Ciqual** (base de composition nutritionnelle de l'ANSES) donne, pour chaque ingrédient, ses valeurs pour 100 g. On joint les deux — quantité × composition, agrégé par recette — pour reconstituer les `Calories`/`Protein`/… de chaque plat. Sur ce volume, le solveur Z3 explore l'espace intelligemment là où l'énumération de la section 4 (coût O(n³) ici) deviendrait infaisable : un argument de plus pour la modélisation déclarative.

## 8. Tableau recapitulatif des approches

| Approche | Technique | Avantage | Limite |
|----------|-----------|----------|--------|
| Index Z3.Linq | `Symbols<int,int,int>` | Syntaxe LINQ naturelle | Contraintes index-only |
| Énumération | Boucles C# + filtres | Simple a comprendre | Pas scalable (O(n^3)) |
| Booléenne Z3 | `BoolExpr` + `MkITE` | Data fusion native | Syntaxe bas niveau |
| Dichotomie | `Push/Pop` + binary search | Optimisation stable | O(log n) appels solver |
| **Hiérarchique `int[][]`** | `NewTheorem<WeeklyPlan>` + `Distinct` | **Structure pure LINQ** | Contraintes linéaires seulement |

## 9. La feature ressuscitée : `CollectionHandling` (mode Array vs Constants)

Jusqu'ici, ce notebook utilisait le fork `Z3.Linq` charge depuis NuGet. Depuis le **14/06/2026**, le fork embarque une enum `CollectionHandling` qui était **définie mais jamais câblée** (dead code). Elle est désormais **vivante** : le même code LINQ peut résoudre un CSP en modélisant les collections de deux façons différentes.

| Mode | Modélisation Z3 | Avantage pédagogique |
|------|----------------|----------------------|
| **`Array`** (défaut) | Une seule `ArrayExpr` Z3 + `Select`/`Store` (théorie des tableaux) | Compact, supporte les index symboliques |
| **`Constants`** | Un constant Z3 par element (`TotalCal_0`, `TotalCal_1`, ...) | « un symbole = une inconnue », lisible dans le modèle |

Pour illustrer concretement, modelisons une **assiette de 3 plats** (entree, plat, dessert) comme un `int[]` de couts, et resolvons un mini-théorème dans les **deux modes** avec le **même** code. C'est la preuve que `CollectionHandling` est fonctionnelle bout-en-bout.

In [8]:
// Demonstration CollectionHandling : un même théorème résolu en mode Array puis Constants.
// Plate = assiette de 3 couts (entree, plat, dessert). On impose : chaque cout dans [100, 800],
// et les 3 couts distincts (un menu varie ne repet pas le même prix).

public class Plate
{
    public int[] Costs { get; set; } = new int[3];
}

public static Theorem<Plate> BuildPlate(Z3Context ctx)
{
    var t = ctx.NewTheorem<Plate>();
    for (int i = 0; i < 3; i++)
    {
        int i1 = i;
        t = t.Where(p => p.Costs[i1] >= 100 && p.Costs[i1] <= 800);
    }
    // 3 couts distincts = menu varie
    t = t.Where(p => Z3Methods.Distinct(p.Costs[0], p.Costs[1], p.Costs[2]));
    return t;
}

var ctxArray = new Z3Context { DefaultCollectionHandling = CollectionHandling.Array };
var ctxConst = new Z3Context { DefaultCollectionHandling = CollectionHandling.Constants };

Console.WriteLine($"Mode Array     : {ctxArray.DefaultCollectionHandling}");
Console.WriteLine($"Mode Constants : {ctxConst.DefaultCollectionHandling}");

var solArray = BuildPlate(ctxArray).Solve();
var solConst = BuildPlate(ctxConst).Solve();

Console.WriteLine("\nSolution (mode Array)     : " + string.Join(", ", solArray.Costs));
Console.WriteLine("Solution (mode Constants) : " + string.Join(", ", solConst.Costs));

Mode Array     : Array


Mode Constants : Constants



Solution (mode Array)     : 100, 102, 101


Solution (mode Constants) : 100, 101, 102


### Interpretation — dead code ressuscite

Le **même** code `BuildPlate` résout le CSP dans les deux modes. La seule différence est la configuration du `Z3Context` :

- **Mode Array** : `Cells`/`Costs` est une seule `ArrayExpr` Z3. L'acces `Costs[i]` genere un `Select(arr, i)`. Compact.
- **Mode Constants** : chaque element `Costs[i]` est un constant Z3 nomme (`Costs_0`, `Costs_1`, `Costs_2`). Si on inspectait le modèle Z3, on verrait ces 3 symboles explicites = modèle « humain ».

| Aspect | Array | Constants |
|--------|-------|-----------|
| Symboles Z3 créés | 1 (l'array) | N (1 par element) |
| Index symbolique | supporte | non (taille fixe) |
| Lisibilite modèle | - | « un symbole = une inconnue » |

C'est la preuve directe que l'enum `CollectionHandling` — historiquement du dead code défini mais jamais câble — est désormais **fonctionnelle bout-en-bout** (construction d'environnement → visite des contraintes → extraction du modèle) dans les deux modes.

---

## Exercices

Les exercices suivants vous demandent d'etendre le modèle du planificateur de repas.

### Exercice 1 : Menu équilibré avec contrainte de glucides

Ajoutez une contrainte sur les **glucides totaux** (entre 40g et 80g) au modèle d'optimisation (section 6). Le menu optimal doit toujours minimiser le cout, mais satisfaire cette contrainte supplementaire.

**Indice** : Ajoutez un `IntExpr totalCarbs` avec le même pattern `MkITE` que `totalCal` et `totalProt`, puis ajoutez les contraintes `MkGe` / `MkLe`.

In [9]:
// Exercice 1 : ajoutez la contrainte 40g <= glucides <= 80g au modèle d'optimisation
// Reprenez le code de la section 6 et ajoutez totalCarbs

// TODO etudiant : implementer la contrainte de glucides
// int minCarbs = 40, maxCarbs = 80;
// IntExpr totalCarbs = ...
// optimizer.Add(z3ctx.MkGe(totalCarbs, z3ctx.MkInt(minCarbs)));
// optimizer.Add(z3ctx.MkLe(totalCarbs, z3ctx.MkInt(maxCarbs)));

Console.WriteLine("Exercice a completer : contrainte de glucides");

Exercice a completer : contrainte de glucides


### Exercice 2 : Planificateur de repas multi-jours

Etendez le modèle pour planifier un menu sur **2 jours** (6 repas : 2 entrees, 2 plats, 2 desserts) sans répétition. Contrainte : chaque aliment ne peut etre sélectionné qu'**au maximum 1 fois**.

**Etape 1** : Creez 6 groupes de variables booléennes (ou utilisez des paires jour/catégorie).

**Etape 2** : Ajoutez la contrainte de non-répétition (un aliment ne peut pas etre vrai dans 2 groupes différents).

**Indice** : Pour chaque aliment, la somme de ses apparitions dans les différents repas doit etre <= 1.

In [10]:
// Exercice 2 : planificateur multi-jours sans répétition

// TODO etudiant : créer les variables pour 2 jours de menus
// Indice : utilisez un dictionnaire (jour, catégorie) -> BoolExpr[]
// Pour la non-répétition : pour chaque aliment, sum(apparitions) <= 1

Console.WriteLine("Exercice a completer : planificateur multi-jours");

Exercice a completer : planificateur multi-jours


### Exercice 3 : Ajout d'un plat supplementaire (boisson)

Ajoutez une catégorie **"boisson"** a la base de données (3 boissons) et modifiez le modèle d'optimisation pour inclure exactement 1 boisson par repas. Verifiez que les contraintes nutritionnelles sont toujours satisfaites.

**Etape 1** : Ajoutez 3 boissons a `foodDatabase`.

**Etape 2** : Ajoutez le groupe "boisson" dans les contraintes `ExactlyOne`.

**Etape 3** : Re-executez l'optimisation et comparez le nouveau menu optimal.

In [11]:
// Exercice 3 : ajout d'une catégorie boisson

// TODO etudiant : ajouter 3 boissons a foodDatabase
// Exemples : eau (0 kcal), jus d'orange (110 kcal, 25g glucides), vin (85 kcal)
// TODO etudiant : ajouter la contrainte ExactlyOne pour la catégorie boisson

Console.WriteLine("Exercice a completer : ajout de boissons");

Exercice a completer : ajout de boissons


### Exercice 4 : Permutation complète des plats (théorème hiérarchique)

Reprenez le modèle `WeeklyPlan` de la section 7 et forcez une **permutation complète** des plats : sur les 3 jours, les 3 plats servis doivent être **tous distincts** (chaque plat servi au plus une fois).

**Indice** : utilisez `Z3Methods.Distinct(p.Plan[0][1], p.Plan[1][1], p.Plan[2][1])` — c'est le même marqueur LINQ que pour les lignes/colonnes d'un Sudoku (notebook 04). Attention à remplacer la contrainte de *montée en gamme* (`p.Plan[j][1] < p.Plan[j+1][1]`) par cette contrainte de *permutation* (`Distinct`) : les deux ne sont pas compatibles simultanément (la permutation autorise n'importe quel ordre, la montée en gamme l'impose croissant).

In [12]:
// Exercice 4 : permutation complète des plats sur 3 jours (Distinct global sur la colonne des plats)
// Reprenez le théorème WeeklyPlan de la section 7 et remplacez la montée en gamme par une permutation.

// TODO etudiant : construire ctx.NewTheorem<WeeklyPlan>() avec les bornes de catégorie (section 7)
// TODO etudiant : remplacer les contraintes p.Plan[j][1] < p.Plan[j+1][1] par :
//   Z3Methods.Distinct(p.Plan[0][1], p.Plan[1][1], p.Plan[2][1])
// TODO etudiant : résoudre et afficher le plan (chaque plat servi au plus une fois)

Console.WriteLine("Exercice a completer : permutation des plats");

Exercice a completer : permutation des plats


---

## Conclusion

Ce notebook a explore la **modélisation hiérarchique** avec Z3 :

1. **Data fusion** : injection de données nutritionnelles dans les expressions Z3 via `MkITE`
2. **Variables booléennes** : sélection d'aliments avec contraintes de cardinalite (exactement 1 par catégorie)
3. **Optimisation par dichotomie** : `Push/Pop` + binary search pour trouver le menu le moins cher
4. **Comparaison des approches** : LINQ index, énumération, modèle booléen, dichotomie

### Points clés a retenir

| Concept | Implementation Z3 |
|---------|-------------------|
| Sélection binaire | `BoolExpr` + exclusion mutuelle |
| Data fusion | `MkITE(variable, valeur, 0)` |
| Contrainte de somme | Agregation `MkAdd` des `MkITE` |
| Optimisation | Dichotomie avec `Push/Pop` |

**Navigation** : [Index Z3](./README.md) | [Notebook précédent (Nested Arrays)](04_Nested_Arrays_2D.ipynb)